In [ ]:
from __future__ import annotations

import hashlib
import re
import zipfile
from pathlib import Path


TASK_COUNT = 400
TASK_RE = re.compile(r"^task(\d{3})\.onnx$")

WORK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("working")
WORK_ROOT.mkdir(parents=True, exist_ok=True)

SUBMISSION_ZIP = WORK_ROOT / "submission.zip"
OVERRIDES = [{"task_id": "task191", "sha10": "722fceefad", "stage": "old_6285_current_valid"}, {"task_id": "task158", "sha10": "0272248663", "stage": "compress_rewrite"}, {"task_id": "task182", "sha10": "6b44786b3f", "stage": "compress_rewrite"}, {"task_id": "task025", "sha10": "a0187f4259", "stage": "compress_rewrite"}, {"task_id": "task387", "sha10": "8d2da09220", "stage": "staticized_repair"}, {"task_id": "task129", "sha10": "bc11e9e80b", "stage": "new_public_source"}, {"task_id": "task150", "sha10": "3713f1d83d", "stage": "gemma4_block"}, {"task_id": "task155", "sha10": "9b6ad9025b", "stage": "gemma4_block"}, {"task_id": "task322", "sha10": "f43610dda9", "stage": "gemma4_block"}, {"task_id": "task338", "sha10": "b35b8a9c78", "stage": "gemma4_block"}, {"task_id": "task364", "sha10": "45105a4fd4", "stage": "gemma4_block"}, {"task_id": "task295", "sha10": "32ce72bb19", "stage": "gemma4_block"}, {"task_id": "task216", "sha10": "8736d30a18", "stage": "gemma4_block"}]

def has_base_tasks(bundle: Path) -> bool:
    return (bundle / "base_submission.zip").exists() or (bundle / "base_submission").is_dir()


def has_overrides(bundle: Path) -> bool:
    return (bundle / "overrides").is_dir() or (bundle / "overrides.zip").exists()


def prepare_zipped_override_bundle(bundle: Path) -> Path:
    prepared = WORK_ROOT / "prepared_asset_bundle"
    prepared.mkdir(parents=True, exist_ok=True)

    if (bundle / "base_submission.zip").exists():
        (prepared / "base_submission.zip").write_bytes((bundle / "base_submission.zip").read_bytes())
    elif (bundle / "base_submission").is_dir():
        base_dir = prepared / "base_submission"
        base_dir.mkdir(parents=True, exist_ok=True)
        for source in sorted((bundle / "base_submission").glob("task*.onnx")):
            (base_dir / source.name).write_bytes(source.read_bytes())

    overrides_dir = prepared / "overrides"
    overrides_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(bundle / "overrides.zip") as zf:
        for entry in zf.infolist():
            if entry.is_dir():
                continue
            name = Path(entry.filename).name
            if re.match(r"^task\d{3}\.onnx$", name):
                (overrides_dir / name).write_bytes(zf.read(entry))
    return prepared


def find_asset_bundle() -> Path:
    search_roots = [Path.cwd()]
    if Path("/kaggle/input").exists():
        search_roots.append(Path("/kaggle/input"))

    for root in search_roots:
        candidates = [root] + [path for path in root.rglob("*") if path.is_dir()]
        for bundle in candidates:
            if has_base_tasks(bundle) and (bundle / "overrides").is_dir():
                return bundle
            if has_base_tasks(bundle) and (bundle / "overrides.zip").exists():
                return prepare_zipped_override_bundle(bundle)

    raise FileNotFoundError(
        "Could not find asset bundle with base_submission plus overrides. "
        "Attach the companion Kaggle Dataset first."
    )


ASSET_DIR = find_asset_bundle()
print(f"Asset bundle: {ASSET_DIR}")


In [ ]:
def sha10(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()[:10]


def task_name(task_id: int) -> str:
    return f"task{task_id:03d}.onnx"


def valid_task_file(name: str) -> bool:
    match = TASK_RE.match(Path(name).name)
    return bool(match and 1 <= int(match.group(1)) <= TASK_COUNT)


In [ ]:
def read_zip_tasks(zip_path: Path) -> dict[str, bytes]:
    tasks: dict[str, bytes] = {}
    with zipfile.ZipFile(zip_path) as zf:
        for entry in zf.infolist():
            if entry.is_dir():
                continue
            name = Path(entry.filename).name
            if valid_task_file(name):
                tasks[name] = zf.read(entry)
    return tasks


def read_dir_tasks(dir_path: Path) -> dict[str, bytes]:
    return {
        path.name: path.read_bytes()
        for path in sorted(dir_path.glob("task*.onnx"))
        if valid_task_file(path.name)
    }


if (ASSET_DIR / "base_submission.zip").exists():
    base_tasks = read_zip_tasks(ASSET_DIR / "base_submission.zip")
else:
    base_tasks = read_dir_tasks(ASSET_DIR / "base_submission")

if len(base_tasks) != TASK_COUNT:
    raise ValueError(f"Base zip has {len(base_tasks)} task files, expected {TASK_COUNT}")

print(f"Loaded base tasks: {len(base_tasks)}")
print(f"Loaded overrides: {len(OVERRIDES)}")


In [ ]:
final_tasks = dict(base_tasks)
applied_rows = []

for row in OVERRIDES:
    name = f"{row['task_id']}.onnx"
    override_path = ASSET_DIR / "overrides" / name
    data = override_path.read_bytes()
    actual_sha10 = sha10(data)
    if actual_sha10 != row["sha10"]:
        raise ValueError(f"{name}: expected {row['sha10']}, got {actual_sha10}")

    before_sha10 = sha10(final_tasks[name])
    final_tasks[name] = data
    applied_rows.append(
        {
            **row,
            "baseline_sha10": before_sha10,
            "applied_sha10": actual_sha10,
            "bytes": len(data),
        }
    )

print(f"Applied overrides: {len(applied_rows)}")
for row in applied_rows:
    print(f"{row['task_id']}: {row['baseline_sha10']} -> {row['applied_sha10']} ({row['stage']})")


In [ ]:
expected_names = {task_name(task_id) for task_id in range(1, TASK_COUNT + 1)}
if set(final_tasks) != expected_names:
    missing = sorted(expected_names - set(final_tasks))
    extra = sorted(set(final_tasks) - expected_names)
    raise ValueError(f"Bad final task set. missing={missing[:5]} extra={extra[:5]}")

with zipfile.ZipFile(SUBMISSION_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for task_id in range(1, TASK_COUNT + 1):
        name = task_name(task_id)
        zf.writestr(name, final_tasks[name])

print(f"Wrote {SUBMISSION_ZIP} ({SUBMISSION_ZIP.stat().st_size:,} bytes)")


In [ ]:
# Final zip structure audit. This catches the most common submission packaging errors.
with zipfile.ZipFile(SUBMISSION_ZIP) as zf:
    entries = [entry.filename for entry in zf.infolist() if not entry.is_dir()]

bad_paths = [name for name in entries if "/" in name or not valid_task_file(name)]
missing = sorted(expected_names - set(entries))
extra = sorted(set(entries) - expected_names)

print(f"Zip entries: {len(entries)}")
print(f"Bad paths: {bad_paths[:5]}")
print(f"Missing: {missing[:5]}")
print(f"Extra: {extra[:5]}")

assert len(entries) == TASK_COUNT
assert not bad_paths
assert not missing
assert not extra
